# Minimal AlphaZero on Tic-Tac-Toe: MCTS as a Policy-Improvement Operator

*A small, self-contained, CPU-only teaching notebook — sibling to the simple RL notebooks.*

This notebook implements **AlphaZero** end-to-end on **Tic-Tac-Toe**, in pure
`numpy` / `torch`. No `gym`, no external RL libraries. Everything fits in one file
and runs in a few minutes on a Colab CPU.

## The AlphaZero recipe (one sentence per ingredient)

1. **A neural net** $f_\theta(s) = (\mathbf{p}, v)$ maps a board state $s$ to a
   **policy prior** $\mathbf{p}$ (a probability over moves) and a scalar **value**
   $v \in [-1, 1]$ (predicted game outcome from the current player's view).
2. **MCTS** (Monte-Carlo Tree Search) uses $f_\theta$ as a guide to search ahead,
   producing a *refined* move distribution $\boldsymbol{\pi}$ from visit counts.
3. **Self-play** plays the net against itself, using MCTS to pick moves and
   recording $(s, \boldsymbol{\pi}, z)$ where $z$ is the eventual game outcome.
4. **Training** fits the net to the search results:
   move toward MCTS's $\boldsymbol{\pi}$, and toward the realized outcome $z$.
5. **Iterate.** The improved net makes MCTS sharper; sharper MCTS makes better
   training targets; repeat.

## The one idea to take away

> **MCTS is a policy-improvement operator.**

Given the network's prior policy $\mathbf{p}$, a fixed budget of look-ahead search
turns it into a *strictly better* policy $\boldsymbol{\pi}$ (the visit-count
distribution). We then *distill* that better policy back into the network via
supervised learning. This is exactly **policy iteration**: MCTS = improvement,
network training = projection/evaluation. The search **amplifies the prior** — a
weak network + search beats the weak network alone, and the gap is the training
signal.

## PUCT — how the search picks moves

At a node $s$, MCTS repeatedly descends by choosing the action maximizing the
**PUCT** score (Predictor + Upper Confidence bound applied to Trees):

$$a^\* = \arg\max_a \; \underbrace{Q(s,a)}_{\text{exploit: mean value}}
   \; + \; \underbrace{c_{\text{puct}}\, P(s,a)\,
   \frac{\sqrt{\sum_b N(s,b)}}{1 + N(s,a)}}_{\text{explore: prior-weighted bonus}}$$

- $Q(s,a)$ — mean value backed up through edge $(s,a)$ so far.
- $P(s,a)$ — the **network prior** for action $a$ (this is the "predictor").
- $N(s,a)$ — visit count of edge $(s,a)$; the bonus shrinks as we revisit.
- $c_{\text{puct}}$ — exploration temperature.

Early on, the prior $P$ dominates (we trust the net). As counts grow, $Q$ takes
over (we trust what we found). No random rollouts: a leaf is evaluated by the
**network's value head** directly — that is the AlphaZero twist over classic MCTS.

## Why Tic-Tac-Toe?

It is **solved**: with perfect play the game is a **draw**. So a well-trained
AlphaZero agent should **essentially never lose**, draw against a perfect
(minimax) opponent, and **crush a random opponent**. That gives us a crisp,
checkable success criterion — perfect for a teaching demo.

## Pointers

- Notion: [MCTS & AlphaZero](https://app.notion.com/p/37f95008d766812c8eebfe55b31c7751)
- Notion: [Model-Based RL](https://app.notion.com/p/37f95008d76681ed845fe254161d6608)

> **Connection to molecular search ↔ learning.** The same search-amplifies-prior
> loop underlies model-based molecular design: a learned proposal/prior over the
> next edit or building block is *amplified* by tree/beam search over a reward
> (docking score, synthesizability, binding), the high-value rollouts are
> distilled back into the proposal net, and you iterate. Tic-Tac-Toe is the
> 9-cell sandbox for that exact idea.


In [ ]:
# === Imports & global configuration =========================================
import time, math, random
from collections import defaultdict
import numpy as np

# Torch is optional for *reading* the notebook, but required to actually train.
# On Colab it is pre-installed. On a bare machine: pip install torch
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    HAS_TORCH = True
except Exception as e:  # pragma: no cover
    HAS_TORCH = False
    print("WARNING: torch not importable ->", e)
    print("The TTT environment + MCTS still run on a uniform net; training is skipped.")

import matplotlib
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# CONFIG — sized for a few minutes on a Colab CPU. Tweak freely.
# ---------------------------------------------------------------------------
class Config:
    # --- self-play / training loop ---
    iterations        = 30     # outer AlphaZero iterations
    selfplay_games    = 25     # self-play games generated per iteration
    train_epochs      = 4      # passes over the replay buffer per iteration
    batch_size        = 64

    # --- MCTS ---
    mcts_sims         = 50     # simulations per move (search budget)
    c_puct            = 1.5    # PUCT exploration constant
    dirichlet_alpha   = 0.3    # root Dirichlet noise concentration
    dirichlet_eps     = 0.25   # mix weight for root noise (0 disables)
    temp_moves        = 4      # # opening plies sampled with temperature=1
                               # (after this, play greedily by visit count)

    # --- network ---
    hidden            = 64     # hidden width of the MLP trunk
    lr                = 1e-3
    weight_decay      = 1e-4
    buffer_size       = 6000   # replay buffer capacity (states)

    # --- evaluation ---
    eval_games        = 100    # games vs random opponent per eval
    eval_every        = 3      # evaluate every k iterations (plus first/last)

    # --- reproducibility ---
    seed              = 0

cfg = Config()

def set_seed(s):
    random.seed(s); np.random.seed(s)
    if HAS_TORCH:
        torch.manual_seed(s)

set_seed(cfg.seed)
print("torch available:", HAS_TORCH)
print("Config:", {k: v for k, v in vars(Config).items() if not k.startswith("_")})


## 1. The Tic-Tac-Toe environment

We represent the board as a length-9 `int8` array, row-major:

```
 index layout        a sample position
  0 | 1 | 2           X | . | O
 ---+---+---         ---+---+---
  3 | 4 | 5           . | X | .
 ---+---+---         ---+---+---
  6 | 7 | 8           O | . | X
```

**Sign convention (this is the part that bites people).** We store the board in
**absolute** marks: `+1` for player X, `-1` for player O, `0` for empty. We track
whose turn it is with `player ∈ {+1, -1}`.

For the network we use a **canonical encoding**: the board is always shown *from
the perspective of the player to move*, by multiplying the board by `player`.
So "my pieces" are always `+1` and "opponent pieces" are always `-1`. This makes
the net player-agnostic and is what makes the **value sign convention** clean:
`value` and `z` are always *"how good is this for the player to move"*.


In [ ]:
# === Tic-Tac-Toe: a tiny, dependency-free environment =======================
#
# State = (board, player)
#   board  : np.int8[9], values in {+1 (X), -1 (O), 0 (empty)}  -- ABSOLUTE marks
#   player : +1 or -1, whose turn it is
#
# All functions are pure (no hidden global state) so MCTS can freely copy states.

WIN_LINES = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),   # rows
    (0, 3, 6), (1, 4, 7), (2, 5, 8),   # cols
    (0, 4, 8), (2, 4, 6),              # diagonals
]

def ttt_init():
    """Return the empty board and the first player (X = +1)."""
    return np.zeros(9, dtype=np.int8), 1

def ttt_legal_moves(board):
    """Indices of empty cells (legal moves)."""
    return np.where(board == 0)[0]

def ttt_legal_mask(board):
    """Boolean[9] mask, True where a move is legal."""
    return board == 0

def ttt_step(board, player, action):
    """Apply `action` (cell index) for `player`. Returns (new_board, next_player).
    Does NOT mutate the input board."""
    assert board[action] == 0, f"illegal move {action} on {board.tolist()}"
    nb = board.copy()
    nb[action] = player
    return nb, -player

def ttt_winner(board):
    """Return +1 if X has three in a row, -1 if O does, else 0 (no winner yet)."""
    for (a, b, c) in WIN_LINES:
        s = board[a] + board[b] + board[c]
        if s == 3:
            return 1
        if s == -3:
            return -1
    return 0

def ttt_is_terminal(board):
    """Game over? True if someone won or the board is full (draw)."""
    return ttt_winner(board) != 0 or not np.any(board == 0)

def ttt_outcome(board, player):
    """Outcome of a TERMINAL board *from `player`'s perspective*:
       +1 win, -1 loss, 0 draw. (Caller must ensure the board is terminal.)"""
    w = ttt_winner(board)
    if w == 0:
        return 0.0                  # draw (or full board)
    return 1.0 if w == player else -1.0

def ttt_encode(board, player):
    """Canonical, player-relative encoding for the network.

    We multiply by `player` so the side to move always sees its own pieces as +1.
    Two planes: [my-pieces (0/1), opponent-pieces (0/1)] -> float32[18].
    (A single signed plane works too; two planes are a touch easier to learn.)"""
    rel = (board * player).astype(np.float32)       # +1 = mine, -1 = theirs
    mine = (rel == 1).astype(np.float32)
    opp  = (rel == -1).astype(np.float32)
    return np.concatenate([mine, opp])              # shape (18,)

def ttt_render(board):
    """Pretty-print a board to a string."""
    sym = {1: "X", -1: "O", 0: "."}
    rows = []
    for r in range(3):
        rows.append(" " + " | ".join(sym[int(board[3*r + c])] for c in range(3)))
    return ("\n" + "-"*11 + "\n").join(rows)

# --- quick sanity checks (these also run during validation) -----------------
def _ttt_self_test():
    b, p = ttt_init()
    assert p == 1 and b.sum() == 0
    # X plays a winning top row: 0,1,2 with O at 3,4 in between
    b = np.array([1, 1, 1, -1, -1, 0, 0, 0, 0], dtype=np.int8)
    assert ttt_winner(b) == 1, "X should win on top row"
    assert ttt_is_terminal(b)
    assert ttt_outcome(b, 1) == 1.0 and ttt_outcome(b, -1) == -1.0
    # O wins a column 0,3,6
    b = np.array([-1, 1, 1, -1, 1, 0, -1, 0, 0], dtype=np.int8)
    assert ttt_winner(b) == -1, "O should win on left column"
    assert ttt_outcome(b, -1) == 1.0 and ttt_outcome(b, 1) == -1.0
    # A full draw board: X O X / X O O / O X X
    b = np.array([1, -1, 1, 1, -1, -1, -1, 1, 1], dtype=np.int8)
    assert ttt_winner(b) == 0 and ttt_is_terminal(b)
    assert ttt_outcome(b, 1) == 0.0
    # encoding: player to move always sees own pieces as +1 in plane 0
    b = np.array([1, 0, 0, 0, -1, 0, 0, 0, 0], dtype=np.int8)
    enc_X = ttt_encode(b, 1)   # X to move: X-cell(0) -> mine
    enc_O = ttt_encode(b, -1)  # O to move: O-cell(4) -> mine
    assert enc_X[0] == 1.0 and enc_X[9 + 4] == 1.0
    assert enc_O[4] == 1.0 and enc_O[9 + 0] == 1.0
    return True

print("TTT self-test:", _ttt_self_test())
print("Empty board:\n" + ttt_render(ttt_init()[0]))


## 2. The policy/value network $f_\theta$

A tiny MLP with a shared trunk and two heads:

- **Policy head** → 9 logits, one per cell. We **mask illegal moves** to $-\infty$
  before the softmax, so the net only ever puts probability on legal cells.
- **Value head** → a single scalar squashed by `tanh` into $[-1, 1]$, interpreted
  as *"expected outcome for the player to move"*.

Input is the canonical 18-dim encoding from `ttt_encode`. Small on purpose:
Tic-Tac-Toe has only 5478 reachable states, so 64 hidden units is plenty.


In [ ]:
# === Policy / Value network =================================================
if HAS_TORCH:

    class AlphaZeroNet(nn.Module):
        """f_theta(state) -> (policy_logits[9], value scalar in [-1,1])."""
        def __init__(self, in_dim=18, hidden=64, n_actions=9):
            super().__init__()
            self.trunk = nn.Sequential(
                nn.Linear(in_dim, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
            )
            self.policy_head = nn.Linear(hidden, n_actions)
            self.value_head  = nn.Linear(hidden, 1)

        def forward(self, x):
            h = self.trunk(x)
            logits = self.policy_head(h)            # raw, UNmasked logits
            value  = torch.tanh(self.value_head(h)) # scalar in [-1, 1]
            return logits, value.squeeze(-1)

    @torch.no_grad()
    def net_infer(net, board, player):
        """Run the net on a single (board, player) and return
        (prior over 9 cells with illegal moves zeroed & renormalized, value float).
        This is the function MCTS calls to expand a leaf."""
        net.eval()
        x = torch.from_numpy(ttt_encode(board, player)).float().unsqueeze(0)
        logits, value = net(x)
        logits = logits.squeeze(0).numpy()
        mask = ttt_legal_mask(board)               # bool[9]
        # Mask illegal moves: set their logit to -inf so softmax gives them 0.
        neg_inf = np.finfo(np.float32).min
        logits = np.where(mask, logits, neg_inf)
        logits = logits - logits.max()             # numerical stability
        exp = np.exp(logits) * mask                # zero out illegal explicitly
        prior = exp / (exp.sum() + 1e-12)
        return prior, float(value.item())

    print("AlphaZeroNet defined.")
else:
    # ---- Fallback so the rest of the notebook is still runnable for reading ----
    class _UniformNet:
        """A stand-in 'network' that returns a uniform legal prior and value 0.
        Lets MCTS run without torch (it becomes vanilla count-based MCTS)."""
        def __call__(self, *a, **k):
            raise RuntimeError("no torch")

    def net_infer(net, board, player):
        mask = ttt_legal_mask(board).astype(np.float32)
        prior = mask / mask.sum()
        return prior, 0.0

    print("torch missing -> using a UNIFORM-prior net_infer (MCTS still works).")


## 3. MCTS with PUCT (the policy-improvement operator)

One **simulation** = one descent from the root to a leaf, an evaluation, and a
backup:

1. **Select.** From the root, repeatedly pick the child maximizing PUCT until we
   reach a node that has not been expanded (a leaf), or a terminal state.
2. **Expand & evaluate.** If the leaf is non-terminal, call the network to get the
   prior $\mathbf{p}$ and value $v$; create the node with those priors. If it is
   terminal, the leaf value is the exact game outcome.
3. **Backup.** Propagate the leaf value up the path, **flipping its sign at every
   step** because players alternate: a position good for me is bad for my
   opponent who chose the move leading here.

After all simulations, the **visit counts** $N(s,a)$ at the root define the
improved policy $\boldsymbol{\pi}(a) \propto N(s,a)^{1/\tau}$. We optionally add
**Dirichlet noise** to the root prior so self-play explores openings.

**The sign convention, stated once and obeyed everywhere:** every node stores
values *from the perspective of the player to move at that node*. A child's value
is the negative of how it looks to the parent. We implement this with a single
`value = -value` flip as we walk back up.


In [ ]:
# === Monte-Carlo Tree Search (PUCT, net-guided, sign-flipping backup) ========
#
# Node identity = canonical state key (board bytes + player). Because TTT is small
# and we rebuild the tree each move, a dict-of-nodes keyed by state is simplest.

class MCTSNode:
    """A search node for one (board, player) state."""
    __slots__ = ("board", "player", "is_terminal", "terminal_value",
                 "prior", "children", "N", "W", "Q", "legal")
    def __init__(self, board, player):
        self.board   = board
        self.player  = player
        self.is_terminal = ttt_is_terminal(board)
        self.terminal_value = ttt_outcome(board, player) if self.is_terminal else None
        self.legal   = ttt_legal_moves(board)         # array of legal cell indices
        self.prior   = None                            # P(s,a) over 9 cells (set on expand)
        self.children = {}                             # action -> child state key
        self.N = np.zeros(9, dtype=np.float64)         # visit counts per action
        self.W = np.zeros(9, dtype=np.float64)         # total value per action
        self.Q = np.zeros(9, dtype=np.float64)         # mean value per action

def _state_key(board, player):
    return (board.tobytes(), player)

class MCTS:
    """Runs `n_sims` PUCT simulations from a root and returns visit counts."""
    def __init__(self, net, c_puct=1.5):
        self.net = net
        self.c_puct = c_puct
        self.nodes = {}     # state_key -> MCTSNode (the search tree)

    def _get_node(self, board, player):
        key = _state_key(board, player)
        node = self.nodes.get(key)
        if node is None:
            node = MCTSNode(board, player)
            self.nodes[key] = node
        return node

    def _expand(self, node):
        """Attach a network prior to a (non-terminal) leaf and return its value
        from the node's player's perspective."""
        prior, value = net_infer(self.net, node.board, node.player)
        node.prior = prior
        return value

    def run(self, root_board, root_player, n_sims,
            dirichlet_alpha=0.0, dirichlet_eps=0.0):
        """Build/extend the tree and return (visit_counts[9], root_node)."""
        self.nodes = {}
        root = self._get_node(root_board, root_player)
        if not root.is_terminal and root.prior is None:
            self._expand(root)

        # Optional Dirichlet noise on the ROOT prior (exploration in self-play).
        if dirichlet_eps > 0 and not root.is_terminal:
            legal = root.legal
            noise = np.random.dirichlet([dirichlet_alpha] * len(legal))
            p = root.prior.copy()
            p[legal] = (1 - dirichlet_eps) * p[legal] + dirichlet_eps * noise
            root.prior = p

        for _ in range(n_sims):
            self._simulate(root)

        return root.N.copy(), root

    def _simulate(self, root):
        """One full simulation: select -> expand/evaluate -> backup."""
        path = []                       # list of (node, action) along the descent
        node = root

        while True:
            if node.is_terminal:
                # Leaf value = exact outcome from this node's player's view.
                leaf_value = node.terminal_value
                break
            if node.prior is None:
                # Unexpanded leaf: evaluate with the network.
                leaf_value = self._expand(node)
                break

            # --- SELECT: pick the legal action with max PUCT ---
            a = self._select_action(node)
            path.append((node, a))
            nb, np_player = ttt_step(node.board, node.player, a)
            child = self._get_node(nb, np_player)
            node.children[a] = _state_key(nb, np_player)
            node = child

        # --- BACKUP with sign flip at every ply ---
        # `leaf_value` is from the leaf player's perspective. The parent that
        # chose the move into the leaf is the OPPONENT, so we negate as we ascend.
        value = leaf_value
        for (parent, a) in reversed(path):
            value = -value              # flip to the parent's perspective
            parent.N[a] += 1
            parent.W[a] += value
            parent.Q[a] = parent.W[a] / parent.N[a]

    def _select_action(self, node):
        """PUCT: argmax_a Q(s,a) + c * P(s,a) * sqrt(sum_b N(s,b)) / (1 + N(s,a))."""
        legal = node.legal
        total_N = node.N.sum()
        sqrt_total = math.sqrt(total_N) if total_N > 0 else 1.0  # sqrt(0)=0 -> use 1 so the prior still drives the very first picks
        # Vectorized over legal moves:
        q = node.Q[legal]
        p = node.prior[legal]
        n = node.N[legal]
        u = self.c_puct * p * (math.sqrt(total_N + 1e-8)) / (1.0 + n)
        scores = q + u
        best = legal[int(np.argmax(scores))]
        return best

def mcts_policy(visit_counts, legal_mask, temperature=1.0):
    """Convert visit counts to a policy pi(a) over 9 cells.
       temperature=1 -> proportional to counts; ->0 -> greedy (one-hot on argmax)."""
    counts = visit_counts.copy().astype(np.float64)
    counts[~legal_mask] = 0.0
    if counts.sum() == 0:                       # safety: no sims hit (shouldn't happen)
        pi = legal_mask.astype(np.float64)
        return pi / pi.sum()
    if temperature <= 1e-6:
        pi = np.zeros_like(counts)
        pi[int(np.argmax(counts))] = 1.0
        return pi
    logits = np.log(counts + 1e-12) / temperature
    logits -= logits.max()
    exp = np.exp(logits) * legal_mask
    return exp / exp.sum()

print("MCTS defined.")


## 4. Self-play → training data

The net plays itself. At each move we run MCTS, record the state + the MCTS
visit-count policy $\boldsymbol{\pi}$, then sample/choose a move. When the game
ends, we go back and label every stored state with the **outcome $z$ from the
perspective of the player who was to move at that state** — which is why the
canonical encoding and sign convention pay off: $z \in \{+1, 0, -1\}$ is always
"did the side to move eventually win".

- **Temperature schedule:** the first few plies are sampled with $\tau=1$
  (exploration / opening diversity); afterwards we play greedily ($\tau\to 0$).
- **Root Dirichlet noise** keeps self-play from collapsing onto one opening.


In [ ]:
# === Self-play: generate (state_encoding, pi, z) training tuples =============

def self_play_game(net, cfg):
    """Play ONE game of net-vs-net with MCTS. Returns a list of training samples:
       (encoded_state[18], pi[9], player_to_move) ; z is filled in after the game."""
    board, player = ttt_init()
    history = []          # (encoded_state, pi, player)
    move_count = 0

    while not ttt_is_terminal(board):
        mcts = MCTS(net, c_puct=cfg.c_puct)
        counts, _ = mcts.run(
            board, player, cfg.mcts_sims,
            dirichlet_alpha=cfg.dirichlet_alpha,
            dirichlet_eps=cfg.dirichlet_eps,
        )
        legal_mask = ttt_legal_mask(board)

        # Training target pi: use temperature=1 over visit counts (the improved
        # policy). We store THIS as the supervised target regardless of how we
        # actually move (AlphaZero stores the search policy).
        pi = mcts_policy(counts, legal_mask, temperature=1.0)

        # Record the canonical-encoded state + improved policy + side to move.
        history.append((ttt_encode(board, player), pi.astype(np.float32), player))

        # Choose the actual move: temperature 1 for the opening, then greedy.
        temp = 1.0 if move_count < cfg.temp_moves else 0.0
        move_pi = mcts_policy(counts, legal_mask, temperature=temp)
        action = int(np.random.choice(9, p=move_pi))

        board, player = ttt_step(board, player, action)
        move_count += 1

    # Game over: assign z to each recorded state from THAT state's mover's view.
    # outcome(board, +1) is the final result for X; flip per stored player.
    final_for_X = ttt_outcome(board, 1)        # +1 X wins, -1 O wins, 0 draw
    samples = []
    for (enc, pi, mover) in history:
        z = final_for_X if mover == 1 else -final_for_X
        samples.append((enc, pi, np.float32(z)))
    return samples

def generate_self_play(net, cfg, n_games):
    """Run `n_games` self-play games; return a flat list of (enc, pi, z)."""
    data = []
    for _ in range(n_games):
        data.extend(self_play_game(net, cfg))
    return data

print("Self-play defined.")


## 5. The training step

We fit the network to the search results with the standard AlphaZero loss:

$$\mathcal{L}(\theta) = \underbrace{(v_\theta(s) - z)^2}_{\text{value: MSE}}
   \;-\; \underbrace{\boldsymbol{\pi}^\top \log \mathbf{p}_\theta(s)}_{\text{policy: cross-entropy to the search policy}}
   \;+\; \underbrace{\lambda\,\lVert\theta\rVert^2}_{\text{weight decay}}$$

- The **value loss** pushes $v_\theta$ toward the realized outcome $z$.
- The **policy loss** is cross-entropy from the MCTS visit distribution
  $\boldsymbol{\pi}$ to the network's (masked) softmax $\mathbf{p}_\theta$ —
  i.e. *distill the improved policy back into the prior*.
- We mask illegal moves in the policy term so the net never wastes capacity on
  them.

This is the "evaluation/projection" half of policy iteration; MCTS was the
"improvement" half.


In [ ]:
# === Training step ==========================================================
if HAS_TORCH:

    def masked_log_softmax(logits, legal_mask):
        """log-softmax over only the legal entries (illegal -> 0 probability)."""
        neg_inf = torch.finfo(logits.dtype).min
        logits = torch.where(legal_mask, logits, torch.full_like(logits, neg_inf))
        return F.log_softmax(logits, dim=-1)

    def train_on_buffer(net, optimizer, buffer, cfg):
        """Run `cfg.train_epochs` passes over the replay buffer. Returns dict of
        average losses for logging."""
        net.train()
        states = np.stack([b[0] for b in buffer]).astype(np.float32)   # (N,18)
        pis    = np.stack([b[1] for b in buffer]).astype(np.float32)   # (N,9)
        zs     = np.array([b[2] for b in buffer], dtype=np.float32)    # (N,)
        # Legal mask recoverable from encoding: a cell is legal iff NOT mine and
        # NOT opponent's. enc = [mine(9), opp(9)] -> occupied = mine+opp.
        occupied = (states[:, :9] + states[:, 9:]) > 0.5
        legal    = ~occupied                                           # (N,9)

        S  = torch.from_numpy(states)
        P  = torch.from_numpy(pis)
        Z  = torch.from_numpy(zs)
        Lm = torch.from_numpy(legal)

        N = S.shape[0]
        agg = {"loss": 0.0, "policy": 0.0, "value": 0.0}; n_batches = 0
        for _ in range(cfg.train_epochs):
            perm = torch.randperm(N)
            for i in range(0, N, cfg.batch_size):
                idx = perm[i:i + cfg.batch_size]
                s, target_pi, z, lm = S[idx], P[idx], Z[idx], Lm[idx]

                logits, value = net(s)
                logp = masked_log_softmax(logits, lm)
                # policy cross-entropy: -sum_a pi(a) log p(a)  (averaged over batch)
                policy_loss = -(target_pi * logp).sum(dim=1).mean()
                value_loss  = F.mse_loss(value, z)
                loss = value_loss + policy_loss

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                agg["loss"]   += float(loss.item())
                agg["policy"] += float(policy_loss.item())
                agg["value"]  += float(value_loss.item())
                n_batches += 1
        for k in agg:
            agg[k] /= max(n_batches, 1)
        return agg

    print("Training step defined.")
else:
    def train_on_buffer(net, optimizer, buffer, cfg):
        return {"loss": float("nan"), "policy": float("nan"), "value": float("nan")}
    print("torch missing -> train_on_buffer is a no-op.")


## 6. Opponents & evaluation

To measure progress we need opponents and a way to play the trained agent against
them:

- **Random** opponent — picks uniformly among legal moves. A competent agent
  should win or draw essentially always (near-zero losses).
- **Minimax** opponent — *perfect* play via full game-tree search with memoization
  (TTT is tiny). Against perfect play the best possible result is a **draw**, so a
  fully-trained agent should produce **all draws, zero losses**.

The agent itself plays by running MCTS and choosing the most-visited move
(greedy, $\tau \to 0$, no Dirichlet noise at eval time).


In [ ]:
# === Opponents and evaluation ===============================================

def random_policy(board, player, rng):
    """Uniform random legal move."""
    legal = ttt_legal_moves(board)
    return int(rng.choice(legal))

# ---- Perfect minimax opponent (with memoization over canonical states) ------
_minimax_cache = {}
def minimax_value(board, player):
    """Optimal value of (board, player) for the player to move: +1/0/-1.
       Standard negamax over the (tiny) TTT tree."""
    if ttt_is_terminal(board):
        return ttt_outcome(board, player)
    key = (board.tobytes(), player)
    if key in _minimax_cache:
        return _minimax_cache[key]
    best = -2.0
    for a in ttt_legal_moves(board):
        nb, npl = ttt_step(board, player, a)
        v = -minimax_value(nb, npl)            # opponent's best, negated
        if v > best:
            best = v
        if best == 1.0:                        # can't do better than a win
            break
    best = best + 0.0                          # normalize -0.0 -> 0.0 for clean printing
    _minimax_cache[key] = best
    return best

def minimax_policy(board, player, rng):
    """Pick an optimal move (random tie-break among equally-optimal moves)."""
    best, best_moves = -2.0, []
    for a in ttt_legal_moves(board):
        nb, npl = ttt_step(board, player, a)
        v = -minimax_value(nb, npl)
        if v > best:
            best, best_moves = v, [a]
        elif v == best:
            best_moves.append(a)
    return int(rng.choice(best_moves))

# ---- The trained agent's move: greedy MCTS (most-visited) -------------------
def agent_move(net, board, player, cfg):
    mcts = MCTS(net, c_puct=cfg.c_puct)
    counts, _ = mcts.run(board, player, cfg.mcts_sims,
                         dirichlet_alpha=0.0, dirichlet_eps=0.0)
    legal_mask = ttt_legal_mask(board)
    pi = mcts_policy(counts, legal_mask, temperature=0.0)   # greedy
    return int(np.argmax(pi))

def play_match(net, cfg, opponent_policy, agent_plays_X, rng):
    """Play one game: agent vs opponent. Returns result FROM THE AGENT'S view:
       +1 agent win, 0 draw, -1 agent loss."""
    board, player = ttt_init()
    agent_mark = 1 if agent_plays_X else -1
    while not ttt_is_terminal(board):
        if player == agent_mark:
            a = agent_move(net, board, player, cfg)
        else:
            a = opponent_policy(board, player, rng)
        board, player = ttt_step(board, player, a)
    return ttt_outcome(board, agent_mark)   # +1/0/-1 from agent's perspective

def evaluate(net, cfg, opponent_policy, n_games, seed=12345):
    """Play n_games (agent alternates X/O). Return (win_rate, draw_rate, loss_rate)."""
    rng = np.random.default_rng(seed)
    w = d = l = 0
    for g in range(n_games):
        r = play_match(net, cfg, opponent_policy, agent_plays_X=(g % 2 == 0), rng=rng)
        if   r > 0: w += 1
        elif r < 0: l += 1
        else:       d += 1
    n = max(n_games, 1)
    return w / n, d / n, l / n

print("Opponents + evaluation defined.")
# minimax sanity: from the empty board, optimal value is a DRAW (0).
print("minimax value of empty board (should be 0.0):",
      minimax_value(*ttt_init()))


## 7. The full AlphaZero loop

Putting it together:

```
for each iteration:
    1. SELF-PLAY:  generate games with MCTS; collect (s, pi, z) into a buffer
    2. TRAIN:      fit the net to the buffer (policy CE + value MSE)
    3. EVALUATE:   periodically, win/draw/loss vs random (and minimax)
```

The replay buffer is a sliding window over recent self-play states, so the net
trains on data from its current (and recent) selves. Watch the **loss-vs-random
curve** climb toward all wins/draws and the **loss rate collapse toward zero**.


In [ ]:
# === The full training loop =================================================
from collections import deque

def train_alphazero(cfg, verbose=True):
    """Run the full AlphaZero loop. Returns (net, history-dict)."""
    set_seed(cfg.seed)
    if HAS_TORCH:
        net = AlphaZeroNet(in_dim=18, hidden=cfg.hidden, n_actions=9)
        optimizer = torch.optim.Adam(net.parameters(), lr=cfg.lr,
                                     weight_decay=cfg.weight_decay)
    else:
        net, optimizer = None, None

    buffer = deque(maxlen=cfg.buffer_size)
    hist = {"iter": [], "win": [], "draw": [], "loss": [],
            "mm_win": [], "mm_draw": [], "mm_loss": [],
            "train_loss": [], "policy_loss": [], "value_loss": []}

    def do_eval(it):
        wr, dr, lr_ = evaluate(net, cfg, random_policy, cfg.eval_games)
        mw, md_, ml = evaluate(net, cfg, minimax_policy,
                               max(cfg.eval_games // 5, 20))
        hist["iter"].append(it)
        hist["win"].append(wr);  hist["draw"].append(dr);  hist["loss"].append(lr_)
        hist["mm_win"].append(mw); hist["mm_draw"].append(md_); hist["mm_loss"].append(ml)
        if verbose:
            print(f"  [eval @ iter {it:>2}] vs RANDOM  W/D/L = "
                  f"{wr:.2f}/{dr:.2f}/{lr_:.2f}   |   vs MINIMAX  W/D/L = "
                  f"{mw:.2f}/{md_:.2f}/{ml:.2f}")

    t0 = time.time()
    # Baseline eval at iter 0 (untrained net).
    do_eval(0)
    for it in range(1, cfg.iterations + 1):
        # 1) self-play
        data = generate_self_play(net, cfg, cfg.selfplay_games)
        buffer.extend(data)
        # 2) train
        losses = train_on_buffer(net, optimizer, list(buffer), cfg)
        hist["train_loss"].append(losses["loss"])
        hist["policy_loss"].append(losses["policy"])
        hist["value_loss"].append(losses["value"])
        # 3) periodic eval
        if (it % cfg.eval_every == 0) or (it == cfg.iterations):
            if verbose:
                print(f"iter {it:>2}/{cfg.iterations}  "
                      f"buffer={len(buffer):>4}  "
                      f"loss={losses['loss']:.3f} "
                      f"(pol={losses['policy']:.3f} val={losses['value']:.3f})")
            do_eval(it)

    elapsed = time.time() - t0
    print(f"\\nTotal training time: {elapsed:.1f} s "
          f"({elapsed/60:.1f} min) for {cfg.iterations} iterations.")
    hist["elapsed"] = elapsed
    return net, hist

# ---- RUN IT --------------------------------------------------------------- #
# (Sized by `cfg` above. Expect a few minutes on a Colab CPU.)
trained_net, history = train_alphazero(cfg)


## 8. Did it learn? Win/draw/loss vs a random opponent

If the loop worked, the curve below should show **losses collapsing to ~0** and
wins+draws approaching **1.0** within a handful of iterations — and the
**minimax** curve should be **all draws** (the agent can no longer be beaten by
perfect play, and cannot beat it either, because TTT is a draw).


In [ ]:
# === Plot: win/draw/loss vs random (and minimax) over iterations ============
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

it = history["iter"]
ax = axes[0]
ax.plot(it, history["win"],  "-o", label="win",  color="#2ca02c")
ax.plot(it, history["draw"], "-o", label="draw", color="#1f77b4")
ax.plot(it, history["loss"], "-o", label="loss", color="#d62728")
ax.set_title("Agent vs RANDOM opponent")
ax.set_xlabel("training iteration"); ax.set_ylabel("rate")
ax.set_ylim(-0.03, 1.03); ax.grid(alpha=0.3); ax.legend(loc="center right")

ax = axes[1]
ax.plot(it, history["mm_win"],  "-o", label="win",  color="#2ca02c")
ax.plot(it, history["mm_draw"], "-o", label="draw", color="#1f77b4")
ax.plot(it, history["mm_loss"], "-o", label="loss", color="#d62728")
ax.set_title("Agent vs PERFECT minimax opponent")
ax.set_xlabel("training iteration"); ax.set_ylabel("rate")
ax.set_ylim(-0.03, 1.03); ax.grid(alpha=0.3); ax.legend(loc="center right")

plt.tight_layout(); plt.show()

# Final-number summary
print(f"FINAL vs random : W/D/L = {history['win'][-1]:.2f} / "
      f"{history['draw'][-1]:.2f} / {history['loss'][-1]:.2f}")
print(f"FINAL vs minimax: W/D/L = {history['mm_win'][-1]:.2f} / "
      f"{history['mm_draw'][-1]:.2f} / {history['mm_loss'][-1]:.2f}")
print("Success looks like: ~0 losses vs random, ALL draws vs minimax.")


## 9. Inspecting the search: MCTS visit-count policy on a sample position

Here we take a concrete board, run MCTS with the trained net, and visualize:

- the **network prior** $\mathbf{p}$ (what the net suggests *without* search), vs
- the **MCTS visit distribution** $\boldsymbol{\pi}$ (the *improved* policy after
  search amplifies the prior).

We use a position with an obvious best move: **X to move, can win immediately** by
completing a line. A trained agent should pour almost all its visits onto that
winning cell — and you should see search **sharpen** the prior toward it.


In [ ]:
# === Heatmap: network prior vs MCTS visit-count policy ======================
# Sample position: X (=+1) to move and can win at cell 2 (completes top row 0,1,2).
#   X | X | .          <- play cell 2 to win
#  ---+---+---
#   O | O | .
#  ---+---+---
#   . | . | .
sample_board = np.array([1, 1, 0,
                         -1, -1, 0,
                         0,  0, 0], dtype=np.int8)
sample_player = 1
print("Sample position (X to move; winning move = cell 2):\n")
print(ttt_render(sample_board), "\n")

prior, value = net_infer(trained_net, sample_board, sample_player)
mcts = MCTS(trained_net, c_puct=cfg.c_puct)
counts, _root = mcts.run(sample_board, sample_player, cfg.mcts_sims,
                         dirichlet_alpha=0.0, dirichlet_eps=0.0)
pi = mcts_policy(counts, ttt_legal_mask(sample_board), temperature=1.0)

print(f"Network value estimate for this position (X to move): {value:+.3f}  "
      f"(closer to +1 = X expects to win)")
print("Per-cell visit counts:", counts.astype(int).tolist())
print(f"MCTS best move (most visited): cell {int(np.argmax(counts))}  "
      f"(expected: 2)")

def _heat(ax, vec, title):
    grid = vec.reshape(3, 3)
    im = ax.imshow(grid, cmap="viridis", vmin=0, vmax=max(vec.max(), 1e-9))
    ax.set_title(title)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center",
                    color="white" if grid[i, j] < 0.5 * vec.max() else "black")
    ax.set_xticks([]); ax.set_yticks([])
    return im

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
_heat(axes[0], prior, "network prior  p(s,a)")
_heat(axes[1], counts / max(counts.sum(), 1), "MCTS visits (normalized)")
_heat(axes[2], pi, r"improved policy $\pi$ (T=1)")
plt.suptitle("Search amplifies the prior: probability mass concentrates on the winning move (cell 2)")
plt.tight_layout(); plt.show()


## 10. A demo game vs the trained agent

Finally, a short printed game: the **trained agent (X)** vs the **perfect minimax
player (O)**. Against perfect play the expected result is a **draw** — watch the
agent defend correctly and refuse to lose.


In [ ]:
# === Printed demo game: trained agent (X) vs perfect minimax (O) ============
def demo_game(net, cfg, seed=7):
    rng = np.random.default_rng(seed)
    board, player = ttt_init()
    ply = 0
    print("Trained agent = X ; perfect minimax = O\n")
    while not ttt_is_terminal(board):
        if player == 1:
            a = agent_move(net, board, player, cfg)
            who = "Agent(X)"
        else:
            a = minimax_policy(board, player, rng)
            who = "Minimax(O)"
        board, player = ttt_step(board, player, a)
        ply += 1
        print(f"ply {ply}: {who} -> cell {a}")
        print(ttt_render(board))
        print()
    w = ttt_winner(board)
    result = "DRAW" if w == 0 else ("X (agent) WINS" if w == 1 else "O (minimax) WINS")
    print("Result:", result, " <- expected DRAW if the agent is trained")
    return result

_ = demo_game(trained_net, cfg)


## 11. Takeaways

- **MCTS is a policy-improvement operator.** A fixed search budget turns the
  network's prior $\mathbf{p}$ into a better policy $\boldsymbol{\pi}$ (visit
  counts). Training distills $\boldsymbol{\pi}$ back into the net — *policy
  iteration* where **search = improvement** and **supervised fit = evaluation**.
- **Search amplifies the prior.** Even a mediocre net + a little look-ahead plays
  far above the net's raw policy; the *gap* between $\mathbf{p}$ and
  $\boldsymbol{\pi}$ is exactly the learning signal. As the net improves, less
  search is needed for the same strength.
- **No rollouts, no human data.** Leaves are evaluated by the **value head**, not
  random playouts — the model *is* the simulator's evaluator. This is the
  **model-based RL** flavor of AlphaZero: a learned value/policy guiding planning.
- **Sign conventions matter.** Everything is stored *from the player-to-move's
  perspective*; the single sign flip in MCTS backup and the per-mover labeling of
  $z$ in self-play are what make a two-player game trainable with one network.
- **Sanity from a solved game.** Tic-Tac-Toe is a forced draw, so the unambiguous
  success signal is **~0 losses vs random and all draws vs minimax** — which the
  loop above should reach in minutes on a CPU.

### Connections

- Notion — [MCTS & AlphaZero](https://app.notion.com/p/37f95008d766812c8eebfe55b31c7751):
  the theory hub for PUCT, the self-play loop, and policy improvement.
- Notion — [Model-Based RL](https://app.notion.com/p/37f95008d76681ed845fe254161d6608):
  AlphaZero as planning with a learned model/evaluator.

### From a 9-cell board to molecules

The identical loop powers **model-based molecular design**: a learned prior over
the next chemical edit / building block is **amplified by tree search** against a
reward (docking, binding affinity, synthesizability); high-reward search rollouts
are **distilled back** into the proposal network; iterate. Swap the TTT
transition model for a chemistry environment and the value head for a property
predictor, and this notebook becomes a sketch of search-guided generative design.
Tic-Tac-Toe is the minimal sandbox for that search ↔ learning idea.
